# Macro-Credit Risk Stress Testing Engine
## Cuaderno 01: Análisis Exploratorio, Pruebas de Raíz Unitaria (ADF/KPSS) y Correlación Cruzada (CCF)
**Objetivo:** Explorar las series históricas macroeconómicas de BCRP y la morosidad de consumo de la SBS, verificar estacionariedad conjunta y seleccionar los rezagos óptimos de transmisión.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.ingestion import DataIngestionPipeline
from src.features import StationarityTester, FinancialTransformations, CCFLagSelector, MulticollinearityController

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 1. Ingesta y Carga de Datos Consolidados (BCRP + SBS)

In [ ]:
raw_path = PROJECT_ROOT / 'data/raw/macro_npl_raw.csv'
if not raw_path.exists():
    pipeline = DataIngestionPipeline(raw_dir=PROJECT_ROOT / 'data/raw')
    df_raw = pipeline.run(use_cache=True)
else:
    df_raw = pd.read_csv(raw_path, index_col='date', parse_dates=True)

print(f'Dimensiones del panel: {df_raw.shape[0]} meses x {df_raw.shape[1]} variables')
df_raw.tail()

### 2. Inspección Visual de Series Macroeconómicas y Morosidad

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()

series_labels = [
    ('npl_consumer', 'Morosidad Consumo SBS (%)', '#dc2626'),
    ('policy_rate', 'Tasa de Política Monetaria BCRP (%)', '#2563eb'),
    ('inflation_12m', 'Inflación IPC Lima Metropolitana 12m (%)', '#d97706'),
    ('fx_rate', 'Tipo de Cambio (S/ por US$)', '#059669'),
    ('gdp_index', 'Índice PBI Global Desestacionalizado', '#7c3aed')
]

for i, (col, label, color) in enumerate(series_labels):
    axes[i].plot(df_raw.index, df_raw[col], color=color, lw=1.8)
    axes[i].set_title(label, fontsize=11, fontweight='bold')
    axes[i].grid(True, alpha=0.3)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

### 3. Pruebas Conjuntas de Raíz Unitaria (ADF + KPSS)

In [ ]:
print('--- DIAGNÓSTICO EN NIVELES CRUDOS ---')
stat_raw = StationarityTester.analyze_panel(df_raw)
print(stat_raw.to_string())

print('\n--- DIAGNÓSTICO EN SERIES TRANSFORMADAS ---')
df_trans = FinancialTransformations.apply_standard_macro_transformations(df_raw)
stat_trans = StationarityTester.analyze_panel(df_trans)
print(stat_trans.to_string())

### 4. Selección de Rezagos Óptimos vía Correlación Cruzada (CCF)

In [ ]:
target_var = 'npl_diff'
macro_vars = ['gdp_growth_yoy', 'policy_rate_diff', 'inflation_diff', 'fx_depreciation_mom']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, var_name in enumerate(macro_vars):
    sel = CCFLagSelector.select_optimal_lag(df_trans[target_var], df_trans[var_name], max_lags=12)
    summary = sel.summary_df
    ax = axes[i]
    ax.bar(summary['Lag_k'], summary['Correlation'], color='#0284c7', alpha=0.7)
    ax.axhline(summary['Bartlett_CI'].iloc[0], color='red', linestyle='--', label='Bandas Bartlett 95%')
    ax.axhline(-summary['Bartlett_CI'].iloc[0], color='red', linestyle='--')
    ax.axvline(sel.optimal_lag, color='green', linestyle=':', lw=2, label=f'Lag k*={sel.optimal_lag}')
    ax.set_title(f'CCF: {var_name} vs {target_var} (Corr={sel.correlation:+.3f})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Rezago k (meses)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### 5. Control de Multicolinealidad (VIF)

In [ ]:
lag_matrix, _ = CCFLagSelector.build_optimal_lag_matrix(df_trans[target_var], df_trans[macro_vars], max_lags=12)
vifs = MulticollinearityController.calculate_vif(lag_matrix.dropna(), add_constant=True)
print('Variance Inflation Factors (VIF < 5.0):')
print(vifs)